# 📒 5. 간단한 예시: 계산기 만들기

> 지금까지 배운 Tkinter 위젯·레이아웃·이벤트 처리를 모두 활용하여 **계산기 앱**을 단계별로 만들어 봅니다.

**학습 목표**
- `StringVar` 로 화면(표시창)을 만들고 업데이트하는 방법
- `grid()` 레이아웃으로 버튼 격자를 구성하는 방법
- 버튼 클릭 이벤트(콜백)를 하나의 함수로 처리하는 방법
- 사칙연산과 `C`(지우기), `=`(계산) 동작을 구현하는 방법

**최종 결과물**
```
┌──────────────┐
│   0          │  ← 표시창(StringVar)
├──────────────┤
│ 7 8 9  C     │
│ 4 5 6  +     │
│ 1 2 3  -     │
│ 0 .  =  ×    │  ← 버튼 격자(grid)
│         ÷    │
└──────────────┘
```


---
## 1단계: 화면(표시창) 만들기 — `StringVar`

계산기의 **표시창**은 사용자가 입력한 숫자와 연산 결과를 보여주는 영역입니다.

- `tk.StringVar()` : 텍스트 값을 저장하고, 값이 바뀌면 연결된 위젯이 자동으로 갱신되는 **특수 변수**
- `Label` 의 `textvariable` 옵션에 `StringVar`를 연결하면, 변수값만 바꿔도 화면이 업데이트됩니다.

> 💡 `text="..."` 대신 `textvariable=변수`를 쓰면, **코드 어디서든 변수.set("새값")** 으로 화면을 바꿀 수 있습니다.


In [ ]:
# ┌─────────────────────────────────────────────┐
# │ 1단계: 표시창(StringVar)만 있는 최소 계산기  │
# └─────────────────────────────────────────────┘
import tkinter as tk

root = tk.Tk()
root.title("계산기 - 1단계")
root.geometry("300x200")

# 1) 화면에 표시될 텍스트를 담는 특수 변수
display = tk.StringVar()
display.set("0")          # 처음에는 0 으로 시작

# 2) 표시창 Label (오른쪽 정렬, 큰 글씨)
screen = tk.Label(root, textvariable=display,
                  font=("맑은 고딕", 24), bg="#222", fg="#0f0",
                  anchor="e", padx=12, pady=12)
screen.pack(fill="x")

root.mainloop()


---
## 2단계: 버튼 격자 만들기 — `grid()`

숫자 버튼과 연산자 버튼을 **바둑판 모양(grid)** 으로 배치합니다.

- `grid(row=행, column=열)` : 위젯을 격자 칸에 배치
- 버튼은 `tk.Button(부모, text="7", command=콜백, width=5, height=2)` 형태
- 이 단계에서는 아직 클릭 동작은 비워두고 **배치만** 확인합니다.

**버튼 배치도**
```
행0:  7   8   9   C
행1:  4   5   6   +
행2:  1   2   3   -
행3:  0   .   =   ×
행4:  (빈)        ÷
```


In [ ]:
# ┌──────────────────────────────────────────┐
# │ 2단계: 버튼 격자 배치 (클릭 동작은 아직)  │
# └──────────────────────────────────────────┘
import tkinter as tk

root = tk.Tk()
root.title("계산기 - 2단계")
root.geometry("300x320")

display = tk.StringVar()
display.set("0")

# 표시창
screen = tk.Label(root, textvariable=display,
                  font=("맑은 고딕", 24), bg="#222", fg="#0f0",
                  anchor="e", padx=12, pady=12)
screen.grid(row=0, column=0, columnspan=4, sticky="ew")

# 버튼에 들어갈 문자열 (한 줄이 한 행)
buttons = [
    ["7", "8", "9", "C"],
    ["4", "5", "6", "+"],
    ["1", "2", "3", "-"],
    ["0", ".", "=", "×"],
    ["",   "",  "", "÷"],
]

# 격자에 버튼 배치
for r, row_vals in enumerate(buttons, start=1):
    for c, txt in enumerate(row_vals):
        if txt == "":              # 빈 칸은 버튼을 만들지 않음
            continue
        btn = tk.Button(root, text=txt, font=("맑은 고딕", 16),
                        width=5, height=2)
        btn.grid(row=r, column=c, padx=2, pady=2)

root.mainloop()


---
## 3단계: 버튼 클릭 핸들러 만들기

버튼을 누르면 **화면의 글자가 바뀌도록** 콜백 함수를 연결합니다.

- 모든 버튼이 **하나의 `on_click(btn_text)` 함수**를 공유
- 숫자/`.` 버튼 → 현재 화면 글자 뒤에 이어붙이기
- `C` → `"0"` 으로 초기화
- 이 단계에서는 연산자/`=` 버튼도 일단 **화면에 글자만 표시** (계산은 4단계에서)

> 💡 `lambda t=txt: on_click(t)` — `t=txt` 처럼 기본값으로 고정해야 각 버튼이 **자기 글자**를 넘깁니다.


In [ ]:
# ┌────────────────────────────────────────────┐
# │ 3단계: 클릭 핸들러 — 화면에 글자 표시/초기화 │
# └────────────────────────────────────────────┘
import tkinter as tk

root = tk.Tk()
root.title("계산기 - 3단계")
root.geometry("300x320")

display = tk.StringVar()
display.set("0")

screen = tk.Label(root, textvariable=display,
                  font=("맑은 고딕", 24), bg="#222", fg="#0f0",
                  anchor="e", padx=12, pady=12)
screen.grid(row=0, column=0, columnspan=4, sticky="ew")

def on_click(btn_text):
    """버튼 클릭 처리 (이 단계에서는 화면 표시만)"""
    current = display.get()

    if btn_text == "C":
        display.set("0")               # 지우기 → 0
    elif btn_text == "=":
        pass                            # 4단계에서 계산 구현
    else:
        # 첫 글자가 0 일 때는 0을 지우고 새로 쓰기
        if current == "0" and btn_text != ".":
            display.set(btn_text)
        else:
            display.set(current + btn_text)

buttons = [
    ["7", "8", "9", "C"],
    ["4", "5", "6", "+"],
    ["1", "2", "3", "-"],
    ["0", ".", "=", "×"],
    ["",   "",  "", "÷"],
]

for r, row_vals in enumerate(buttons, start=1):
    for c, txt in enumerate(row_vals):
        if txt == "":
            continue
        btn = tk.Button(root, text=txt, font=("맑은 고딕", 16),
                        width=5, height=2,
                        command=lambda t=txt: on_click(t))
        btn.grid(row=r, column=c, padx=2, pady=2)

root.mainloop()


---
## 4단계: 사칙연산 구현하기

이제 `+`, `-`, `×`, `÷` 버튼이 **실제로 동작**하도록 만듭니다.

**핵심 아이디어**
1. 사용자가 숫자를 입력 → **첫 번째 피연산자** 저장
2. 연산자 버튼 클릭 → 첫 번째 피연산자를 저장하고, 연산자를 기억, 화면 지우기
3. 두 번째 숫자 입력 → **두 번째 피연산자**
4. `=` 클릭 → 저장된 연산자로 두 수를 계산하여 화면에 표시

**상태 변수 3개**
- `operand1` : 첫 번째 숫자 (없으면 `None`)
- `operator` : 선택된 연산자 (`"+"`, `"-"`, `"×"`, `"÷"` 또는 `None`)
- `new_operand` : `True`이면 다음 숫자 입력 시 **화면을 새로 시작** (연산자/`=` 직후)

> 💡 `new_operand` 플래그가 없으면 연산자 클릭 후 화면을 `"0"`으로 바꾸고 나서
> 새 숫자가 `0` 뒤에 붙어버리는 문제가 생깁니다. 플래그로 "새로 시작" 상태를 명확히 표시하세요.


In [ ]:
# ┌──────────────────────────────────────────────────┐
# │ 4단계: 사칙연산 (+, -, ×, ÷) 구현                │
# └──────────────────────────────────────────────────┘
import tkinter as tk

root = tk.Tk()
root.title("계산기 - 4단계")
root.geometry("300x320")

display = tk.StringVar()
display.set("0")

# 계산 상태를 저장할 변수들
operand1    = None       # 첫 번째 피연산자
operator     = None      # 선택된 연산자
new_operand  = False     # True면 다음 숫자 입력 시 화면을 새로 시작

screen = tk.Label(root, textvariable=display,
                  font=("맑은 고딕", 24), bg="#222", fg="#0f0",
                  anchor="e", padx=12, pady=12)
screen.grid(row=0, column=0, columnspan=4, sticky="ew")

def calculate(a, op, b):
    """a op b 계산 결과 반환"""
    if op == "+": return a + b
    if op == "-": return a - b
    if op == "×": return a * b
    if op == "÷":
        if b == 0:
            return "Error: 0으로 나눌 수 없음"
        return a / b

def on_click(btn_text):
    global operand1, operator, new_operand
    current = display.get()

    if btn_text == "C":
        display.set("0")
        operand1    = None
        operator     = None
        new_operand  = False

    elif btn_text in ("+", "-", "×", "÷"):
        # 첫 번째 숫자 저장, 연산자 기억
        operand1   = float(current)
        operator    = btn_text
        new_operand = True          # 다음 숫자는 새로 시작

    elif btn_text == "=":
        if operand1 is not None and operator is not None:
            operand2 = float(current)
            result = calculate(operand1, operator, operand2)
            # 정수 결과면 소수점 떼기
            if isinstance(result, float) and result.is_integer():
                result = int(result)
            display.set(str(result))
            operand1    = None
            operator     = None
            new_operand  = True      # 결과 확인 후 새 입력은 새로 시작

    else:
        # 숫자 또는 소수점 입력
        if new_operand:
            # 연산자/=/ 결과 직후 → 새 숫자 시작
            if btn_text == ".":
                display.set("0.")
            else:
                display.set(btn_text)
            new_operand = False
        else:
            if current == "0" and btn_text != ".":
                display.set(btn_text)
            elif btn_text == "." and "." in current:
                pass                        # 소수점 중복 방지
            else:
                display.set(current + btn_text)

buttons = [
    ["7", "8", "9", "C"],
    ["4", "5", "6", "+"],
    ["1", "2", "3", "-"],
    ["0", ".", "=", "×"],
    ["",   "",  "", "÷"],
]

for r, row_vals in enumerate(buttons, start=1):
    for c, txt in enumerate(row_vals):
        if txt == "":
            continue
        btn = tk.Button(root, text=txt, font=("맑은 고딕", 16),
                        width=5, height=2,
                        command=lambda t=txt: on_click(t))
        btn.grid(row=r, column=c, padx=2, pady=2)

root.mainloop()


---
## 5단계: 완성된 계산기 — 구조 정리 & 연속 연산

4단계의 로직을 그대로 유지하면서, **코드를 함수로 묶고** 레이아웃을 조금 더 다듬은 **최종 버전**입니다.

**개선점**
- `build_gui()` 함수로 GUI 구성을 분리
- `÷` 버튼을 4행 3열 자리로 옮겨 격자를 균일하게
- `=` 버튼은 두 칸(columnspan=2) 너비로 강조
- **연속 연산** 지원: `2 + 3 × 4 =` → `(2+3)×4 = 20`
- 결과가 `Error:` 로 시작하면 더 이상 계산하지 않도록 보호


In [ ]:
# ┌──────────────────────────────────────────────────┐
# │ 5단계: 완성된 계산기 (최종) — 연속 연산 지원      │
# └──────────────────────────────────────────────────┘
import tkinter as tk

# ---------- 계산 로직 ----------
def calculate(a, op, b):
    if op == "+": return a + b
    if op == "-": return a - b
    if op == "×": return a * b
    if op == "÷":
        if b == 0:
            return "Error: 0으로 나눌 수 없음"
        return a / b

# ---------- 상태 ----------
display      = tk.StringVar()
display.set("0")
operand1     = None
operator      = None
new_operand   = False

# ---------- 클릭 핸들러 ----------
def on_click(btn_text):
    global operand1, operator, new_operand
    current = display.get()

    if current.startswith("Error") and btn_text != "C":
        return                       # 에러 상태는 C만 허용

    if btn_text == "C":
        display.set("0")
        operand1    = None
        operator     = None
        new_operand  = False

    elif btn_text in ("+", "-", "×", "÷"):
        if operand1 is None:
            operand1 = float(current)
        else:
            # 연속 연산: 먼저 계산 후 결과를 첫 피연산자로
            operand2 = float(current)
            result = calculate(operand1, operator, operand2)
            if isinstance(result, str):      # Error
                display.set(result)
                operand1 = None
                operator  = None
                new_operand = True
                return
            if isinstance(result, float) and result.is_integer():
                result = int(result)
            display.set(str(result))
            operand1 = float(result)
        operator     = btn_text
        new_operand  = True

    elif btn_text == "=":
        if operand1 is not None and operator is not None:
            operand2 = float(current)
            result = calculate(operand1, operator, operand2)
            if isinstance(result, str):      # Error
                display.set(result)
            else:
                if isinstance(result, float) and result.is_integer():
                    result = int(result)
                display.set(str(result))
            operand1    = None
            operator     = None
            new_operand  = True

    else:
        # 숫자 또는 소수점 입력
        if new_operand:
            if btn_text == ".":
                display.set("0.")
            else:
                display.set(btn_text)
            new_operand = False
        else:
            if current == "0" and btn_text != ".":
                display.set(btn_text)
            elif btn_text == "." and "." in current:
                pass
            else:
                display.set(current + btn_text)

# ---------- GUI 구성 ----------
def build_gui():
    root = tk.Tk()
    root.title("계산기")
    root.geometry("280x340")
    root.resizable(False, False)

    screen = tk.Label(root, textvariable=display,
                      font=("맑은 고딕", 24), bg="#222", fg="#0f0",
                      anchor="e", padx=12, pady=12)
    screen.grid(row=0, column=0, columnspan=4, sticky="ew")

    #        7    8    9    C
    #        4    5    6    +
    #        1    2    3    -
    #        0    .    =    ×
    #                       ÷
    layout = [
        [("7",1),("8",1),("9",1),("C",1)],
        [("4",1),("5",1),("6",1),("+",1)],
        [("1",1),("2",1),("3",1),("-",1)],
        [("0",1),(".",1),("=",2),("×",1)],
        [("÷",1)],
    ]

    for r, row in enumerate(layout, start=1):
        col = 0
        for txt, span in row:
            btn = tk.Button(root, text=txt, font=("맑은 고딕", 16),
                            width=5*span, height=2,
                            command=lambda t=txt: on_click(t))
            btn.grid(row=r, column=col, columnspan=span, padx=2, pady=2)
            col += span
    return root

root = build_gui()
root.mainloop()


---
## 🎯 정리

이 노트에서 배운 계산기 만들기의 핵심을 한눈에 정리합니다.

| 단계 | 핵심 개념 | 사용한 위젯/메서드 |
|:---:|---|---|
| **1** | 표시창 = `StringVar` + `Label(textvariable=…)` | `tk.StringVar()`, `Label`, `anchor="e"` |
| **2** | 버튼을 격자로 배치 | `tk.Button`, `grid(row, column)` |
| **3** | 하나의 콜백으로 모든 버튼 처리 | `command=lambda t=txt: on_click(t)` |
| **4** | 사칙연산 상태 관리 | `global` 변수 3개(`operand1`, `operator`, `new_operand`) |
| **5** | 구조 정리 & 연속 연산·에러 보호 | 함수 분리, `columnspan`, 에러 가드 |

**기억할 팁**
- `StringVar.set("값")` 한 줄로 화면이 바뀐다 — `Label` 의 `text`를 직접 고칠 필요 없음
- `lambda t=txt: …` 패턴으로 각 버튼이 **자기 문자**를 콜백에 전달
- 계산기는 **상태 기계**: 현재 입력 중인 숫자 + 저장된 연산자 + 저장된 첫 피연산자 + "새 입력 시작?" 플래그
- `new_operand` 플래그로 연산자/`=` 클릭 직후에 **새 숫자가 깨끗이 시작**되도록 한다
- 0으로 나누기, 소수점 중복, 에러 후 입력 같은 **예외 상황**을 반드시 처리


---
## 🎯 연습 문제

완성된 계산기에 다음 **3가지 기능**을 추가해 보세요. 각 문제는 독립적으로 풀 수 있습니다.

| 문제 | 기능 | 난이도 |
|:---:|---|:---:|
| **1** | **백스페이스(⌫)** 버튼 추가 | ★☆☆ |
| **2** | **키보드 입력** 지원 (숫자·연산자·Enter·Backspace) | ★★☆ |
| **3** | **퍼센트(%)** 버튼 추가 | ★★☆ |

아래 코드 셀에 각 문제의 답을 작성하세요. 정답은 셀 아래의 `✅ 정답 보기` 를 펼쳐 확인하세요.


In [ ]:
# ┌────────────────────────────────────────────┐
# │ 연습 문제 1: 백스페이스(⌫) 버튼 추가        │
# └────────────────────────────────────────────┘
# 힌트:
#   - "⌫" 버튼을 새로 만들어 grid 에 배치
#   - on_click 안에 elif btn_text == "⌫": 분기 추가
#   - 현재 화면 문자열에서 마지막 한 글자를 지우기
#   - 모두 지워지면 "0" 으로 되돌리기

# TODO: on_click 에 백스페이스 동작을 추가해 보세요
pass

# ┌────────────────────────────────────────────┐
# │ 연습 문제 2: 키보드 입력 지원               │
# └────────────────────────────────────────────┘
# 힌트:
#   - root.bind("<Key>", on_key) 로 키보드 이벤트 연결
#   - event.char 가 "0"~"9", "+", "-", "*", "/" 면 on_click 호출
#   - event.keysym == "Return" → on_click("=")
#   - event.keysym == "BackSpace" → on_click("⌫")
#   - "*" 은 "×", "/" 는 "÷" 로 변환

# TODO: on_key 함수를 작성하고 root.bind 를 추가해 보세요
pass

# ┌────────────────────────────────────────────┐
# │ 연습 문제 3: 퍼센트(%) 버튼 추가            │
# └────────────────────────────────────────────┘
# 힌트:
#   - "%" 버튼을 새로 만들어 grid 에 배치
#   - on_click 안에 elif btn_text == "%": 분기 추가
#   - 현재 화면 숫자를 100으로 나눈 값으로 표시
#   - 예: 50 → "%" 클릭 → 0.5

# TODO: on_click 에 퍼센트 동작을 추가해 보세요
pass


---
<details><summary>✅ 정답 보기 (클릭)</summary>

### 문제 1: 백스페이스(⌫) 정답

```python
# on_click 함수 안에 아래 분기를 추가
elif btn_text == "⌫":
    current = display.get()
    if len(current) > 1:
        display.set(current[:-1])      # 마지막 한 글자 제거
    else:
        display.set("0")               # 한 글자만 남았으면 0
```

버튼 배치 시 `"⌫"` 문자열을 layout 목록에 추가하면 됩니다.
예: `("⌫",1)` 을 첫 행에 추가하고 `"C"` 옆에 배치.

### 문제 2: 키보드 입력 정답

```python
def on_key(event):
    """키보드 입력을 on_click 으로 변환"""
    key_map = {"*": "×", "/": "÷"}
    ch = event.char
    sym = event.keysym

    if ch and ch in "0123456789.+-":
        on_click(ch)
    elif ch in key_map:
        on_click(key_map[ch])
    elif sym == "Return":
        on_click("=")
    elif sym == "BackSpace":
        on_click("⌫")
    elif sym == "Escape":
        on_click("C")

# build_gui 안에서 root 반환 전에 연결
root.bind("<Key>", on_key)
```

`root.bind("<Key>", on_key)` 한 줄로 모든 키 입력이 `on_click` 과 같은 경로를 타게 됩니다.

### 문제 3: 퍼센트(%) 정답

```python
# on_click 함수 안에 아래 분기를 추가
elif btn_text == "%":
    current = display.get()
    if current not in ("0", "") and not current.startswith("Error"):
        value = float(current) / 100
        if value.is_integer():
            value = int(value)
        display.set(str(value))
```

`50` 입력 후 `%` 클릭 → `0.5` 가 표시됩니다.
연산자와 조합하면 `200 + 10%` 같은 계산도 가능합니다.

</details>
